In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import re
import faiss
import string
from sklearn.metrics.pairwise import cosine_similarity
from transformers import pipeline
from sentence_transformers import SentenceTransformer

In [2]:
financial_df = pd.read_csv('Financial-QA-10k.csv')[['question', 'answer', 'context']]
financial_df

,question,answer,context
0,What area did NVIDIA initially focus on before...,NVIDIA initially focused on PC graphics.,"Since our original focus on PC graphics, we ha..."
1,What are some of the recent applications of GP...,Recent applications of GPU-powered deep learni...,Some of the most recent applications of GPU-po...
2,What significant invention did NVIDIA create i...,NVIDIA invented the GPU in 1999.,Our invention of the GPU in 1999 defined moder...
3,How does NVIDIA's platform strategy contribute...,NVIDIA's platform strategy brings together har...,"NVIDIA has a platform strategy, bringing toget..."
4,What does NVIDIA's CUDA programming model enable?,NVIDIA's CUDA programming model opened the par...,With our introduction of the CUDA programming ...
...,...,...,...
6995,What was the interest rate for the 5.400% Seni...,5.400%,The 5.400% Senior Notes due in 2028 have an in...
6996,What changes were made to the LVSC Revolving C...,The Fourth Amendment to the LVSC Revolving Cre...,"On January 30, 2023, LVSC entered into amendme..."
6997,What was the increase in interest expense for ...,The interest expense increased by $30 million ...,"Following the downgrades, each series of the o..."
6998,What are the new leverage and interest coverag...,"As of January 2024, the new leverage ratio sho...",The amended and restated facility agreement wi...


In [3]:
programming_df1 = pd.read_csv('train2.csv')[['Title', 'Body', 'Tags']]
programming_df2 = pd.read_csv('valid2.csv')[['Title', 'Body', 'Tags']]

In [4]:
programming_df = programming_df1.merge(programming_df2, how='outer')
programming_df

,Title,Body,Tags
0,! in input buffer in comparison in if statement,I am not clear with the usage of ! in input bu...,<c><string><gcc><char>
1,!= operator condition usage in c#,<p>I know that in C++ you can do this kind of ...,<c#><c++><operators><conditional-statements>
2,""" missing the right parenthisis"" Oracle SQL",` `CREATE TABLE note(\r\nnote_id VARCHAR(7)NOT...,<sql><oracle>
3,""" textarea "" multilines PHP/ HTML",I am using php and html codes in same file for...,<php><html>
4,""""" Error in Connection String in Visual Studio...","I am getting error because "" is being applied ...",<c#><asp.net><visual-studio-2013><connection-s...
...,...,...,...
59995,“value $ is not a member of StringContext” - M...,<p>I'm using maven with scala archetype. I'm g...,<scala><apache-spark>
59996,• No instance for (Num [Integer]) arising from...,"{-Hi , can anyone help me here ? -}\r\n\r\nplu...",<haskell><types><functional-programming>
59997,【AdMob】I can show test ads but I can't show re...,"When asking with a stack overflow, there was a...",<c#><unity3d><admob>
59998,【GCP】Cloud Speech-to-Text API Detecting langua...,I would like to activate the function that aut...,<c#><google-cloud-platform><google-speech-api>


In [5]:
Insurance_df = pd.read_csv('test_predictions2.csv')
Insurance_df

,qid,input,predicted_outputs,ground_truth
0,0,Does Medicare Cover Co-Pays?,['Original Medicare Part A & Medicare Part B d...,Original Medicare Part A & Medicare Part B doe...
1,1,Does Auto Insurance Decrease At Age 25?,['Auto insurance policies typically have a red...,Auto insurance policies typically have a reduc...
2,2,Does Auto Insurance Decrease At Age 25?,"[""Your auto insurance coverage certainly does ...",Your auto insurance coverage certainly does n'...
3,3,Does Auto Insurance Decrease At Age 25?,['In California age is not a rating factor . E...,In California age is not a rating factor . Exp...
4,4,Is Health Insurance Elastic Or Inelastic?,"[""I believe that health insurance is neither e...",I believe that health insurance is neither ela...
...,...,...,...,...
3303,3303,Who Sells Immediate Annuities?,['To find an agency that sells immediate annui...,To find an agency that sells immediate annuiti...
3304,3304,What Does Basic Homeowners Insurance Cover?,['A standard homeowners policy should include ...,A standard homeowners policy should include th...
3305,3305,Why does it cost more to insure an employee wh...,"[""That is a great question , with a very simpl...","That is a great question , with a very simple ..."
3306,3306,Why does it cost more to insure an employee wh...,['The workers compensation benefit payable to ...,The workers compensation benefit payable to a ...


# data cleaning

In [6]:
financial_df.isnull().sum(), programming_df.isnull().sum(), Insurance_df.isnull().sum()

(question    2
 answer      2
 context     1
 dtype: int64,
 Title    0
 Body     0
 Tags     0
 dtype: int64,
 qid                  0
 input                0
 predicted_outputs    0
 ground_truth         0
 dtype: int64)

In [7]:
financial_df.dropna(axis=0, inplace=True)

In [8]:
financial_df = financial_df[financial_df['question'].duplicated() == False]
financial_df.reset_index(drop=True, inplace=True)

In [9]:
programming_df = programming_df[programming_df['Title'].duplicated() == False]
programming_df.reset_index(drop=True, inplace=True)

In [10]:
Insurance_df = Insurance_df[Insurance_df['input'].duplicated() == False]
Insurance_df.reset_index(drop=True, inplace=True)

In [15]:

def text_preprocessing_query(text):
    # convert in lower case
    text = text.lower()
    # remove html tags
    text = re.sub(r'<.*?>', '', text)
    # remove punctuations
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)
    # remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def text_preprocessing_answer(text):
    text = re.sub(r'<*?>', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def clean_programming_query(text):
    # Remove only repeated special junk (3+ times)
    text = re.sub(r'([@#$%^&*])\1+', '', text)
    # Remove unwanted symbols but keep programming syntax
    text = re.sub(r'[^a-zA-Z0-9\s.,?!(){}\[\]=+\-*/#@]', '', text)
    return text

In [16]:
financial_df['question'] = financial_df['question'].apply(text_preprocessing_query)
financial_df['context'] = financial_df['context'].apply(text_preprocessing_answer)

C:\Users\Sourav\AppData\Local\Temp\ipykernel_6988\692929845.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  financial_df['question'] = financial_df['question'].apply(text_preprocessing_query)
C:\Users\Sourav\AppData\Local\Temp\ipykernel_6988\692929845.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  financial_df['context'] = financial_df['context'].apply(text_preprocessing_answer)


In [17]:
Insurance_df['input'] = Insurance_df['input'].apply(text_preprocessing_query)
Insurance_df['ground_truth'] = Insurance_df['ground_truth'].apply(text_preprocessing_answer)

C:\Users\Sourav\AppData\Local\Temp\ipykernel_6988\3936716467.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  Insurance_df['input'] = Insurance_df['input'].apply(text_preprocessing_query)
C:\Users\Sourav\AppData\Local\Temp\ipykernel_6988\3936716467.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  Insurance_df['ground_truth'] = Insurance_df['ground_truth'].apply(text_preprocessing_answer)


In [18]:
programming_df['Title'] = programming_df['Title'].apply(clean_programming_query)

C:\Users\Sourav\AppData\Local\Temp\ipykernel_6988\1465060678.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  programming_df['Title'] = programming_df['Title'].apply(clean_programming_query)


In [50]:
# reset index
financial_df = financial_df.reset_index(drop=True)

In [20]:
# hugging face model 
model = SentenceTransformer('all-MiniLM-L6-v2')

C:\Users\Sourav\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [21]:
model

SentenceTransformer(
  (0): Transformer({'max_seq_length': 256, 'do_lower_case': False, 'architecture': 'BertModel'})
  (1): Pooling({'word_embedding_dimension': 384, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
  (2): Normalize()
)

# generate embedding

In [22]:
financial_query_embedding = model.encode(financial_df['question'])
programming_query_embedding= model.encode(programming_df['Title'])
Insurance_query_embedding = model.encode(Insurance_df['input'])

# faiss vector database

In [23]:

#fincanical faiss database
faiss.normalize_L2(financial_query_embedding)
financial_faiss_database = faiss.IndexFlatIP(financial_query_embedding.shape[1])
financial_faiss_database.add(financial_query_embedding)

# programming faiss database
faiss.normalize_L2(programming_query_embedding)
programming_faiss_database = faiss.IndexFlatIP(programming_query_embedding.shape[1])
programming_faiss_database.add(programming_query_embedding)

# insurance faiss database
faiss.normalize_L2(Insurance_query_embedding)
Insurance_faiss_database = faiss.IndexFlatIP(Insurance_query_embedding.shape[1])
Insurance_faiss_database.add(Insurance_query_embedding)

In [24]:
financial_unanswerd_query = []
programming_unanswerd_query = []
Insurance_unanswerd_query = []

# faq function

In [25]:
def faq_match(ques, choose):
    # ques preprocessing
    ques = text_preprocessing_query(ques)
    # generate embedding
    query_embedding = model.encode([ques])
    # apply domain condition (Financial, Programming, Insaurance)

    #for financial
    if choose == 'financial':  
        similarity, index = financial_faiss_database.search(query_embedding, k=1)
        if similarity[0][0] >= 0.85:
            result = f"{financial_df['answer'].loc[index[0][0]]} {financial_df['context'].loc[index[0][0]]}"
            return result
        elif similarity[0][0] >= 60 or similarity[0][0] < 85:
            print('Not sure, are you asking about')
            dis, idx = financial_faiss_database.search(query_embedding, k=3)
            ques1 = financial_df['question'].loc[idx[0][0]]
            ques2 = financial_df['question'].loc[idx[0][1]]
            ques3 = financial_df['question'].loc[idx[0][2]]
            print(f'1. {ques1} \n2. {ques2} \n3. {ques3}')

            user_choice = input('enter your choice 1, 2, 3 : ')
            if user_choice == '1':
                ques1_embedding = model.encode([ques1])
                d, i = financial_faiss_database.search(ques1_embedding, k=1)
                return financial_df['answer'].iloc[i[0][0]]
                
            elif user_choice == '2':
                ques2_embedding = model.encode([ques2])
                d, i = financial_faiss_database.search(ques2_embedding, k=1)
                return financial_df['answer'].iloc[i[0][0]]
                
            elif user_choice == '3':
                ques3_embedding = model.encode([ques3])
                d, i = financial_faiss_database.search(ques3_embedding, k=1)
                return financial_df['answer'].iloc[i[0][0]]
                
            else:
                financial_unanswerd_query.append(ques)
        else:
            financial_unanswerd_query.append(ques)
            return 'No information!'

     # for programming       
    elif choose == 'programming':
        similarity, index = programming_faiss_database.search(query_embedding, k=1)
        if similarity[0][0] >= 0.85:
            return programming_df['Body'].iloc[index[0][0]]
        elif similarity[0][0] >= 60 or similarity[0][0] < 85:
            print('Not sure, are you asking about')
            dis, idx = programming_faiss_database.search(query_embedding, k=3)
            ques1 = programming_df['Title'].loc[idx[0][0]]
            ques2 = programming_df['Title'].loc[idx[0][1]]
            ques3 = programming_df['Title'].loc[idx[0][2]]
            print(f'1. {ques1} \n2. {ques2} \n3. {ques3}')

            user_choice = input('enter your choice 1, 2, 3: ')
            if user_choice == '1':
                ques1_embedding = model.encode([ques1])
                d, i = programming_faiss_database.search(ques1_embedding, k=1)
                ans = programming_df['Body'].iloc[i[0][0]]
                return ans.replace('\\r\\n', '\n'), i
            elif user_choice == '2':
                ques2_embedding = model.encode([ques2])
                d, i = programming_faiss_database.search(ques2_embedding, k=1)
                return programming_df['Body'].iloc[i[0][0]]
            elif user_choice == '3':
                ques3_embedding = model.encode([ques3])
                d, i = programming_faiss_database.search(ques3_embedding, k=1)
                return programming_df['Body'].iloc[i[0][0]]
            else:
                programming_unanswerd_query.append(ques)
        else:
            programming_unanswerd_query.append(ques)
            return 'No information!'
            
    # for insaurance
    elif choose == 'insaurance':
        similarity, index = Insurance_faiss_database.search(query_embedding, k=1)
        if similarity[0][0] >= 0.85:
            return Insurance_df['ground_truth'].iloc[index[0][0]]
        elif similarity[0][0] >= 60 or similarity[0][0] < 85:
            print('Not sure, are you asking about')
            dis, idx = Insurance_faiss_database.search(query_embedding, k=3)
            ques1 = Insurance_df['input'].loc[idx[0][0]]
            ques2 = Insurance_df['input'].loc[idx[0][1]]
            ques3 = Insurance_df['input'].loc[idx[0][2]]
            print(f'1. {ques1} \n2. {ques2} \n3. {ques3}')

            user_choice = input('enter your choice 1, 2, 3: ')
            if user_choice == '1':
                ques1_embedding = model.encode([ques1])
                d, i = Insurance_faiss_database.search(ques1_embedding, k=1)
                return Insurance_df['ground_truth'].iloc[i[0][0]]
            elif user_choice == '2':
                ques2_embedding = model.encode([ques2])
                d, i = Insurance_faiss_database.search(ques2_embedding, k=1)
                return Insurance_df['ground_truth'].iloc[i[0][0]]
            elif user_choice == '3':
                ques3_embedding = model.encode([ques3])
                d, i = Insurance_faiss_database.search(ques3_embedding, k=1)
                return Insurance_df['ground_truth'].iloc[i[0][0]]
            else:
                Insurance_unanswerd_query.append(ques)
        else:
            Insurance_unanswerd_query.append(ques)
            return 'No information!'
    else:
        pass

In [26]:
ans = faq_match('program', 'programming')
for i in ans:
    print(i, end='')

Not sure, are you asking about
1. Please help me with this program 
2. Output of the program 
3. Basic program, but whats the solution?


enter your choice 1, 2, 3:  1


 protected void Page_Load(object sender, EventArgs e)
    {
        

        if (DateTime.Now.Hour < 12)
        {
            lblGreeting.Text = "Good Morning";
            
        }
        else if (DateTime.Now.Hour < 17)
        {
            lblGreeting.Text = "Good Afternoon";
            
        }
        else
        {
            lblGreeting.Text = "Good Evening";
            
        }
        
        
    }
    protected void Button1_Click(object sender, EventArgs e)
    {
       if (DateTime.Now.Date = TextBox2.Text)
       {
           Label2.Text = "Happy Birthday";
       }
       else
       {
           Label2.Text = "Have a nice day";

       }
    }
It gives me error at the last if condition under button 1 click event shows error that cannot implicitly convert type string to system date and time what changes should be done?[[35068]]

In [72]:
faq_match('loan finance', 'financial')

Not sure, are you asking about
1. what financial support mechanisms were implemented for borrowers experiencing financial difficulty as of january 1 2023 
2. what were the proceeds from borrowings and the repayments of borrowings according to the latest available data 
3. what is the companys current position regarding borrowings as of may 31 2023


enter your choice 1, 2, 3 :  2


'The proceeds from borrowings were $2.6 million and the repayments of borrowings were $359.6 million.'

In [29]:
faq_match('how to get money', 'insaurance')

Not sure, are you asking about
1. how can i save money for retirement 
2. how to save a lot of money for retirement 
3. what is a money purchase retirement plan


enter your choice 1, 2, 3:  1


"That is an excellent question ! There are several really good ways to start your retirement planning , and I will tell you that the sooner you start , the easier it is . If your employer offers a retirement plan where they will match your contribution , or match a percentage , do it ! They are giving you `` free '' money . If they offer an HSA -LRB- Health savings Account -RRB- , contribute to that also.By contributing regularly and often to these accounts , you will have a good start to having an affordable retirement . There are definitely other things that you could do , and it would require far more conversation and space than we have here . If you would like more information , feel free to contact me , ok ? Thanks for asking !"

In [58]:
model.save_pretrained('./faq_model')

In [31]:
import joblib

In [32]:
joblib.dump(financial_faiss_database, 'finance_database.pkl')
joblib.dump(programming_faiss_database, 'programming_database.pkl', compress=9)
joblib.dump(Insurance_faiss_database, 'insurance_database.pkl')

['insurance_database.pkl']

In [33]:
joblib.dump(financial_df, 'finance_df.pkl')
joblib.dump(programming_df, 'programming_df.pkl', compress=9)
joblib.dump(Insurance_df, 'insurance_df.pkl')

['insurance_df.pkl']